🔧 1. ollama_client.py

In [ ]:
from ollama import Client

client = Client(host='http://localhost:11434')

def get_client():
    return client


In [ ]:
# Utils
def encode_image_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode()

✍️ 2. story_generator.py

In [ ]:
import os
from ollama_client import get_client

def generate_story(index, config, out_dir="stories"):
    os.makedirs(out_dir, exist_ok=True)
    prompt = f"Write a {config['style']} short story in the genre of {config['genre']}. Make it imaginative and coherent."

    response = get_client().generate(
        model=config["model"],
        prompt=prompt,
        options={
            "temperature": config["temp"],
            "top_k": config["top_k"],
            "top_p": config["top_p"],
            "seed": config.get("seed", 42)
        }
    )

    story = response['response']
    filename = f"{out_dir}/story_{index}_{config['genre'].lower()}.txt"
    with open(filename, "w") as f:
        f.write(story)
    return filename


📝 3. summarizer.py

In [ ]:
import os
from ollama_client import get_client

def summarize_story(story_path, model="llama3.2", out_dir="summaries"):
    os.makedirs(out_dir, exist_ok=True)
    with open(story_path, "r") as f:
        text = f.read()

    prompt = (
        "Summarize the following story using simple sentences. "
        "Keep the summary short and focus on the main plot and message.\n\n"
        f"{text}"
    )

    response = get_client().generate(
        model=model,
        prompt=prompt,
        options={"temperature": 0.4, "top_k": 20, "top_p": 0.7, "seed": 123}
    )

    summary = response["response"].strip()
    filename = os.path.basename(story_path).replace("story_", "summary_")
    with open(f"{out_dir}/{filename}", "w") as f:
        f.write(summary)
    return summary


🧠 4. nli_predictor.py

In [ ]:
import csv
from ollama_client import get_client

def predict_nli(premise, hypothesis, model="llama3.2"):
    prompt = f"""Determine the relationship between the following premise and hypothesis.
If the hypothesis must be true, answer "entailment".
If it must be false, answer "contradiction".
If it neither follows nor conflicts, answer "neutral".

Premise: {premise}
Hypothesis: {hypothesis}
Answer:"""

    response = get_client().generate(model=model, prompt=prompt, options={"temperature": 0.0})
    answer = response["response"].lower()
    if "entail" in answer:
        return "entailment"
    elif "contradict" in answer:
        return "contradiction"
    elif "neutral" in answer:
        return "neutral"
    return "unknown"

def predict_nli_dataset(csv_path, out_path="nli_predictions.csv", model="llama3.2"):
    with open(csv_path) as infile, open(out_path, "w", newline="") as outfile:
        reader = csv.DictReader(infile)
        writer = csv.DictWriter(outfile, fieldnames=["premise", "hypothesis", "prediction"])
        writer.writeheader()

        for row in reader:
            label = predict_nli(row["premise"], row["hypothesis"], model)
            writer.writerow({**row, "prediction": label})


🖼️ 5. image_captioner.py

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from ollama_client import get_client
from utils import encode_image_base64

def caption_image(img_path, model="qwen2.5vl:3b"):
    b64_img = encode_image_base64(img_path)
    response = get_client().generate(
        model=model,
        prompt="Describe this image in one natural sentence.",
        images=[b64_img],
        options={"temperature": 0.5, "top_k": 40}
    )
    return response["response"].strip()

def caption_all_images(image_dir, out_path="captions.txt", workers=4):
    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))]
    results = []

    def process(img_name):
        path = os.path.join(image_dir, img_name)
        return img_name, caption_image(path)

    with ThreadPoolExecutor(max_workers=workers) as executor:
        for img_name, caption in executor.map(process, image_files):
            results.append((img_name, caption))

    with open(out_path, "w") as f:
        for name, cap in results:
            f.write(f"{name}\t{cap}\n")
    return out_path
